In [4]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
#
from pathlib import Path
current_dir = Path.cwd()
ROOT_DIR = current_dir.parent
os.chdir(ROOT_DIR)
from src.distributions import group_data_dist,pie_distribution,target_barchart

SCHEMA_DIR = ROOT_DIR / "schema"
DATA_DIR = ROOT_DIR / "data"

In [3]:
ROOT_DIR

PosixPath('/Users/oyieth/Documents/Python projects/credit_scoring_examples/notebooks')

In [5]:
pd.set_option('display.float_format', '{:.2f}'.format)


# Previous Application Data
From previous application: Installment payments

In [6]:
prev_application = pd.read_csv(DATA_DIR / "previous_application.csv")

installments_payments = pd.read_csv(DATA_DIR /"installments_payments.csv")
pos_cash_balance = pd.read_csv(DATA_DIR /"POS_CASH_balance.csv")
credit_card = pd.read_csv(DATA_DIR /"credit_card_balance.csv")



In [6]:
prev_application.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,1,0.0,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [368]:
features_previous = prev_application[["SK_ID_CURR"]].set_index("SK_ID_CURR").sample(5)
features_previous

""
SK_ID_CURR
290904
101589
292481
410102
269054


In [229]:
def mode_column(df,
               col,
               id_col,
               cols_count,
               cols_sum):
    agg_method = {}
    for item in cols_count:
        agg_method[item] = 'count'
    for item in cols_sum:
        agg_method[item] = 'sum'
    modes_type = df.groupby(by = [id_col, col], as_index = False).agg(agg_method)
    
    modes_type = modes_type.sort_values([id_col]+list(agg_method), ascending= [True]+ [False for item in list(agg_method) ] )
    modes_type['row_num'] = modes_type.groupby(id_col).cumcount() + 1
    modes_type = modes_type[modes_type["row_num"]==1].set_index(id_col)[col]
    return modes_type
#.sort_values(by = ["SK_ID_PREV","AMT_CREDIT"], ascending = [False, False])

In [296]:
def summarize_installments(id_prev_applications,
                          installments_payments = installments_payments):
    # Useful installment data
    installments_payments["pct_payment"] = (installments_payments["AMT_PAYMENT"]/installments_payments["AMT_INSTALMENT"]).replace(np.inf, 0)
    installments_payments.loc[(installments_payments["pct_payment"]<1)&(installments_payments["AMT_INSTALMENT"]>0),"flag_unfullfilled"] = 1
    installments_payments["flag_unfullfilled"] = installments_payments["flag_unfullfilled"].fillna(0)
    installments_payments.loc[(installments_payments["pct_payment"]==0)&(installments_payments["AMT_INSTALMENT"]>0),"flag_unpayment"] = 1
    installments_payments["flag_unpayment"] = installments_payments["flag_unpayment"].fillna(0)
    
    summary_installment = id_prev_applications.set_index("SK_ID_PREV")
    summary_installment["n_versions"] = installments_payments.groupby(by = "SK_ID_PREV").NUM_INSTALMENT_VERSION.nunique()
    summary_installment["total_installments"] = installments_payments.groupby(by = "SK_ID_PREV").NUM_INSTALMENT_NUMBER.max()
    summary_installment["latest_day"] = installments_payments.groupby(by = "SK_ID_PREV").DAYS_INSTALMENT.max()
    summary_installment["first_day"] = installments_payments.groupby(by = "SK_ID_PREV").DAYS_INSTALMENT.min()

    summary_installment["latest_payment"] = installments_payments.groupby(by = "SK_ID_PREV").DAYS_ENTRY_PAYMENT.max() 
    summary_installment["first_payment"] = installments_payments.groupby(by = "SK_ID_PREV").DAYS_ENTRY_PAYMENT.min() 

    summary_installment["max_installment"] = installments_payments.groupby(by = "SK_ID_PREV").AMT_INSTALMENT.max()
    summary_installment["min_installment"] = installments_payments.groupby(by = "SK_ID_PREV").AMT_INSTALMENT.min()
    summary_installment["mean_installment"] = installments_payments.groupby(by = "SK_ID_PREV").AMT_INSTALMENT.mean()
    summary_installment["mean_pay_pct_installment"] = installments_payments.groupby(by = "SK_ID_PREV").pct_payment.mean()
    summary_installment["min_pay_pct_installment"] = installments_payments.groupby(by = "SK_ID_PREV").pct_payment.min()
    summary_installment["max_pay_pct_installment"] = installments_payments.groupby(by = "SK_ID_PREV").pct_payment.max()
    summary_installment["times_unfullfilled"] = installments_payments.groupby(by = "SK_ID_PREV").flag_unfullfilled.sum()
    summary_installment["times_unpayed"] = installments_payments.groupby(by = "SK_ID_PREV").flag_unpayment.sum()
    return summary_installment


In [353]:
def summarize_cash_balance(id_prev_applications,
                          pos_cash_balance = pos_cash_balance):
    pos_summary = id_prev_applications.set_index("SK_ID_PREV")
    pos_summary["n_months"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["MONTHS_BALANCE"].count()

    pos_summary["total_prev_terms"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT"].nunique()
    pos_summary["min_prev_term"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT"].min()
    pos_summary["max_prev_term"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT"].max()
    pos_summary["max_remaining"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT_FUTURE"].max()
    pos_summary["number_status"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["NAME_CONTRACT_STATUS"].nunique()
    pos_summary["total_dpd"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["SK_DPD"].sum()
    pos_summary["total_dpd_def"] = pos_cash_balance.groupby(by = "SK_ID_PREV")["SK_DPD_DEF"].sum()
    pos_summary["flag_dpd"] = pos_summary["total_dpd"].apply(lambda x: 1 if x>0 else 0)
    pos_summary["flag_dpd_def"] = pos_summary["total_dpd"].apply(lambda x: 1 if x>0 else 0)
    latest_month = pos_cash_balance.groupby("SK_ID_PREV", as_index = False)["MONTHS_BALANCE"].max()
    pos_summary["latest_status"] = latest_month.merge(pos_cash_balance[["SK_ID_PREV","MONTHS_BALANCE","NAME_CONTRACT_STATUS"]],
                   on = ["SK_ID_PREV","MONTHS_BALANCE"])[["SK_ID_PREV", "NAME_CONTRACT_STATUS"]
                ].rename(columns = {"NAME_CONTRACT_STATUS":"last_status"}).set_index('SK_ID_PREV')
    return pos_summary

In [360]:
credit_card.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.97,135000,0.00,877.50,0.00,877.50,1700.33,1800.00,1800.00,0.00,0.00,0.00,0.00,1,0.00,1.00,35.00,Active,0,0
1,2582071,363914,-1,63975.56,45000,2250.00,2250.00,0.00,0.00,2250.00,2250.00,2250.00,60175.08,64875.56,64875.56,1.00,1,0.00,0.00,69.00,Active,0,0
2,1740877,371185,-7,31815.22,450000,0.00,0.00,0.00,0.00,2250.00,2250.00,2250.00,26926.42,31460.08,31460.08,0.00,0,0.00,0.00,30.00,Active,0,0
3,1389973,337855,-4,236572.11,225000,2250.00,2250.00,0.00,0.00,11795.76,11925.00,11925.00,224949.29,233048.97,233048.97,1.00,1,0.00,0.00,10.00,Active,0,0
4,1891521,126868,-1,453919.46,450000,0.00,11547.00,0.00,11547.00,22924.89,27000.00,27000.00,443044.40,453919.46,453919.46,0.00,1,0.00,1.00,101.00,Active,0,0


In [370]:
prev_app_detail

,SK_ID_CURR,SK_ID_PREV,n_months_cc,mean_amt_balance,max_amt_balance,min_amt_balance,mean_amt_limit,max_amt_limit,min_amt_limit,mean_amt_drawing_current,max_amt_drawing_current,min_amt_drawing_current,mean_amt_drawing_other_current,max_amt_drawing_other_current,min_amt_drawing_other_current,mean_amt_drawing_pos_current,max_amt_drawing_pos_current,min_amt_drawing_pos_current,mean_min_installment,max_min_installment,min_min_installment,mean_payment,max_paymen,min_payment,mean_payment_total,max_paymen_total,min_payment_total,mean_receivable_principal,max_receivable_principal,min_receivable_principal,mean_receivable,max_receivable,min_receivable,mean_receivable_total,max_receivable_total,min_receivable_total,mean_drawings_current,max_drawings_current,min_drawings_current,mean_drawings_other_current,max_drawings_other_current,min_drawings_other_current,mean_drawings_pos_current,max_drawings_pos_current,min_drawings_pos_current,mean_installments_mature,max_installments_mature,min_installments_mature,total_dpd,total_dpd_def,flag_dpd,flag_dpd_def,latest_status
0,290904,1120684,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
1,290904,1609297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
2,290904,1083477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
3,290904,2098351,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
4,290904,2223555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
5,290904,2768911,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
6,101589,1655460,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
7,101589,1153977,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
8,101589,1632902,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
9,101589,1816729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN


In [366]:
def summarize_credit_card(id_prev_applications,
                          credit_card = credit_card):
    cc_summary = id_prev_applications.set_index("SK_ID_PREV")


    
    cc_summary["n_months_cc"] = credit_card.groupby(by = "SK_ID_PREV")["MONTHS_BALANCE"].count()

    cc_summary["mean_amt_balance"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_BALANCE"].mean()
    cc_summary["max_amt_balance"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_BALANCE"].max()
    cc_summary["min_amt_balance"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_BALANCE"].min()

    cc_summary["mean_amt_limit"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_CREDIT_LIMIT_ACTUAL"].mean()
    cc_summary["max_amt_limit"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_CREDIT_LIMIT_ACTUAL"].max()
    cc_summary["min_amt_limit"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_CREDIT_LIMIT_ACTUAL"].min()

    cc_summary["mean_amt_drawing_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_ATM_CURRENT"].mean()
    cc_summary["max_amt_drawing_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_ATM_CURRENT"].max()
    cc_summary["min_amt_drawing_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_ATM_CURRENT"].min()

    cc_summary["mean_amt_drawing_other_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_OTHER_CURRENT"].mean()
    cc_summary["max_amt_drawing_other_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_OTHER_CURRENT"].max()
    cc_summary["min_amt_drawing_other_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_OTHER_CURRENT"].min()

    cc_summary["mean_amt_drawing_pos_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_POS_CURRENT"].mean()
    cc_summary["max_amt_drawing_pos_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_POS_CURRENT"].max()
    cc_summary["min_amt_drawing_pos_current"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_DRAWINGS_POS_CURRENT"].min()

    cc_summary["mean_min_installment"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_INST_MIN_REGULARITY"].mean()
    cc_summary["max_min_installment"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_INST_MIN_REGULARITY"].max()
    cc_summary["min_min_installment"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_INST_MIN_REGULARITY"].min()

    cc_summary["mean_payment"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_PAYMENT_CURRENT"].mean()
    cc_summary["max_paymen"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_PAYMENT_CURRENT"].max()
    cc_summary["min_payment"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_PAYMENT_CURRENT"].min()

    cc_summary["mean_payment_total"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_PAYMENT_TOTAL_CURRENT"].mean()
    cc_summary["max_paymen_total"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_PAYMENT_TOTAL_CURRENT"].max()
    cc_summary["min_payment_total"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_PAYMENT_TOTAL_CURRENT"].min()

    cc_summary["mean_receivable_principal"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_RECEIVABLE_PRINCIPAL"].mean()
    cc_summary["max_receivable_principal"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_RECEIVABLE_PRINCIPAL"].max()
    cc_summary["min_receivable_principal"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_RECEIVABLE_PRINCIPAL"].min()

    cc_summary["mean_receivable"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_RECIVABLE"].mean()
    cc_summary["max_receivable"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_RECIVABLE"].max()
    cc_summary["min_receivable"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_RECIVABLE"].min()

    cc_summary["mean_receivable_total"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_TOTAL_RECEIVABLE"].mean()
    cc_summary["max_receivable_total"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_TOTAL_RECEIVABLE"].max()
    cc_summary["min_receivable_total"] = credit_card.groupby(by = "SK_ID_PREV")["AMT_TOTAL_RECEIVABLE"].min()

    cc_summary["mean_drawings_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_CURRENT"].mean()
    cc_summary["max_drawings_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_CURRENT"].max()
    cc_summary["min_drawings_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_CURRENT"].min()

    cc_summary["mean_drawings_other_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_OTHER_CURRENT"].mean()
    cc_summary["max_drawings_other_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_OTHER_CURRENT"].max()
    cc_summary["min_drawings_other_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_OTHER_CURRENT"].min()

    cc_summary["mean_drawings_pos_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_POS_CURRENT"].mean()
    cc_summary["max_drawings_pos_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_POS_CURRENT"].max()
    cc_summary["min_drawings_pos_current"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_DRAWINGS_POS_CURRENT"].min()

    cc_summary["mean_installments_mature"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"].mean()
    cc_summary["max_installments_mature"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"].max()
    cc_summary["min_installments_mature"] = credit_card.groupby(by = "SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"].min()
    

    
    cc_summary["total_dpd"] = credit_card.groupby(by = "SK_ID_PREV")["SK_DPD"].sum()
    cc_summary["total_dpd_def"] = credit_card.groupby(by = "SK_ID_PREV")["SK_DPD_DEF"].sum()
    cc_summary["flag_dpd"] = cc_summary["total_dpd"].apply(lambda x: 1 if x>0 else 0)
    cc_summary["flag_dpd_def"] = cc_summary["total_dpd"].apply(lambda x: 1 if x>0 else 0)
    latest_month = credit_card.groupby("SK_ID_PREV", as_index = False)["MONTHS_BALANCE"].max()
    cc_summary["latest_status"] = latest_month.merge(credit_card[["SK_ID_PREV","MONTHS_BALANCE","NAME_CONTRACT_STATUS"]],
                   on = ["SK_ID_PREV","MONTHS_BALANCE"])[["SK_ID_PREV", "NAME_CONTRACT_STATUS"]
                ].rename(columns = {"NAME_CONTRACT_STATUS":"last_status"}).set_index('SK_ID_PREV')
    return cc_summary

In [378]:
def get_previous_application_features(id_loans,
                                     prev_application,
                                     installments_payments = installments_payments,
                                     pos_cash_balance = pos_cash_balance):
    features_previous = id_loans[['SK_ID_CURR']].set_index('SK_ID_CURR')
    features_previous_instlmnt = features_previous.copy()
    features_previous_cash = features_previous.copy()
    features_previous_cc = features_previous.copy()
    # Cleanup prev application
    prev_application["flag_canceled"] = prev_application["NAME_CONTRACT_STATUS"].apply(lambda x: 1 if x=='Canceled' else 0)
    prev_application["flag_refused"] = prev_application["NAME_CONTRACT_STATUS"].apply(lambda x: 1 if x=='Refused' else 0)
    prev_application["flag_unused"] = prev_application["NAME_CONTRACT_STATUS"].apply(lambda x: 1 if x=='Unused offer' else 0)

    prev_application["flag_down_payment"] = prev_application["RATE_DOWN_PAYMENT"].apply(lambda x: 0 if pd.isna(x) else 1)
    prev_application["flag_interest"] = prev_application["RATE_INTEREST_PRIMARY"].apply(lambda x: 0 if pd.isna(x) else 1)
    for item in ['NAME_CASH_LOAN_PURPOSE',
                 'NAME_PAYMENT_TYPE',
                 'NAME_GOODS_CATEGORY',
                 'NAME_PORTFOLIO',
                 'NAME_PRODUCT_TYPE',
                 'NAME_SELLER_INDUSTRY',
                 'NAME_YIELD_GROUP',
                 'PRODUCT_COMBINATION',
                ]:
        prev_application[item] = prev_application[item].replace("XNA", np.nan)
    prev_application["FLAG_LAST_APPL_PER_CONTRACT"] = prev_application["FLAG_LAST_APPL_PER_CONTRACT"].apply(lambda x:1 if x == 'Y' else 0 )


    
    #Numeric columns
    #number of previous aplication
    features_previous["previous_applications"] = prev_application.groupby(by = "SK_ID_CURR")["SK_ID_PREV"].nunique()
    #Diferent types in previous applicacion
    features_previous["n_types_applications"] = prev_application.groupby(by = "SK_ID_CURR")["NAME_CONTRACT_TYPE"].nunique()
    #Different weekday start of application
    features_previous["n_weekdays"] = prev_application.groupby(by = "SK_ID_CURR")['WEEKDAY_APPR_PROCESS_START'].nunique()
    #Different hours when starting previous application
    features_previous["n_hour_start"] = prev_application.groupby(by = "SK_ID_CURR")['HOUR_APPR_PROCESS_START'].nunique()
    #Number of different purposes of application
    features_previous["n_purpose"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_CASH_LOAN_PURPOSE'].nunique()
    #Number of different status for applicacion
    features_previous["n_status"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_CONTRACT_STATUS'].nunique()
    #Number of payment types for previous application
    features_previous["n_payment_type"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_PAYMENT_TYPE'].nunique()
    #Number of code rejections for previous application
    features_previous["n_code_rejection"] = prev_application.groupby(by = "SK_ID_CURR")['CODE_REJECT_REASON'].nunique()
    #Number of type suites for previous application
    features_previous["n_suite"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_TYPE_SUITE'].nunique()
    #Number of client types for previous application
    features_previous["n_client_type"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_CLIENT_TYPE'].nunique()
    #Number of names for goods category for previous application
    features_previous["n_goods_category"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_GOODS_CATEGORY'].nunique()
    #Number of portfolios for previous application
    features_previous["n_portfolio"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_PORTFOLIO'].nunique()
    #Number of product types for previous application
    features_previous["n_product_type"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_PRODUCT_TYPE'].nunique()
    #Number of channel types for previous application
    features_previous["n_channel_type"] = prev_application.groupby(by = "SK_ID_CURR")['CHANNEL_TYPE'].nunique()
    #Number of seller areas for previous application
    features_previous["n_sellerplace_area"] = prev_application.groupby(by = "SK_ID_CURR")['SELLERPLACE_AREA'].nunique()
    #Numer of seller industries for previous application
    features_previous["n_seller_industry"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_SELLER_INDUSTRY'].nunique()
    #Number for yield groups for previous application
    features_previous["n_yield_groups"] = prev_application.groupby(by = "SK_ID_CURR")['NAME_YIELD_GROUP'].nunique()
    #Number of product combinations
    features_previous["n_product_comb"] = prev_application.groupby(by = "SK_ID_CURR")['PRODUCT_COMBINATION'].nunique()

#Amounts: Continuos distributions.mean, mix, max and median. 
    features_previous["amt_annuity_mean"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_ANNUITY"].mean()
    features_previous["amt_annuity_min"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_ANNUITY"].min()
    features_previous["amt_annuity_max"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_ANNUITY"].max()
    features_previous["amt_annuity_median"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_ANNUITY"].median()

    features_previous["amt_application_mean"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_APPLICATION"].mean()
    features_previous["amt_application_min"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_APPLICATION"].min()
    features_previous["amt_application_max"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_APPLICATION"].max()
    features_previous["amt_application_median"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_APPLICATION"].median()

    features_previous["amt_credit_mean"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_CREDIT"].mean()
    features_previous["amt_credit_min"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_CREDIT"].min()
    features_previous["amt_credit_max"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_CREDIT"].max()
    features_previous["amt_credit_median"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_CREDIT"].median()

    features_previous["amt_down_payment_mean"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_DOWN_PAYMENT"].mean()
    features_previous["amt_down_payment_min"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_DOWN_PAYMENT"].min()
    features_previous["amt_down_payment_max"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_DOWN_PAYMENT"].max()
    features_previous["amt_down_payment_median"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_DOWN_PAYMENT"].median()

    features_previous["amt_goods_mean"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_GOODS_PRICE"].mean()
    features_previous["amt_goods_min"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_GOODS_PRICE"].min()
    features_previous["amt_goods_max"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_GOODS_PRICE"].max()
    features_previous["amt_goods_median"] = prev_application.groupby(by = "SK_ID_CURR")["AMT_GOODS_PRICE"].median()

    features_previous["cnt_payment_mean"] = prev_application.groupby(by = "SK_ID_CURR")["CNT_PAYMENT"].mean()
    features_previous["cnt_payment_min"] = prev_application.groupby(by = "SK_ID_CURR")["CNT_PAYMENT"].min()
    features_previous["cnt_payment_max"] = prev_application.groupby(by = "SK_ID_CURR")["CNT_PAYMENT"].max()
    features_previous["cnt_payment_median"] = prev_application.groupby(by = "SK_ID_CURR")["CNT_PAYMENT"].median()

    features_previous["down_payment_mean"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_DOWN_PAYMENT"].mean()
    features_previous["down_payment_min"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_DOWN_PAYMENT"].min()
    features_previous["down_payment_max"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_DOWN_PAYMENT"].max()
    features_previous["down_payment_median"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_DOWN_PAYMENT"].median()

    features_previous["rate_interested_primary_mean"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIMARY"].mean()
    features_previous["rate_interested_primary_min"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIMARY"].min()
    features_previous["rate_interested_primary_max"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIMARY"].max()
    features_previous["rate_interested_primary_median"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIMARY"].median()

    features_previous["rate_intereseted_privileged_mean"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIVILEGED"].mean()
    features_previous["rate_intereseted_privileged_min"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIVILEGED"].min()
    features_previous["rate_intereseted_privileged_max"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIVILEGED"].max()
    features_previous["rate_intereseted_privileged_median"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_INTEREST_PRIVILEGED"].median()


#Days: Min and Max only (Oldest and more recent)

    features_previous["days_decision_min"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_DECISION"].min()
    features_previous["days_decision_max"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_DECISION"].max()
 
    features_previous["days_first_drawing_min"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_FIRST_DRAWING"].min()
    features_previous["days_first_drawing_max"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_FIRST_DRAWING"].max()
	
    features_previous["days_first_due_min"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_FIRST_DUE"].min()
    features_previous["days_first_due_max"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_FIRST_DUE"].max()
	
    features_previous["days_last_due_first_min"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_LAST_DUE_1ST_VERSION"].min()
    features_previous["days_last_due_first_max"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_LAST_DUE_1ST_VERSION"].max()
	
    features_previous["days_last_due_min"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_LAST_DUE"].min()
    features_previous["days_last_due_max"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_LAST_DUE"].max()
	
    features_previous["days_termination_min"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_TERMINATION"].min()
    features_previous["days_termination_max"] = prev_application.groupby(by = "SK_ID_CURR")["DAYS_TERMINATION"].max()

    #Mean
    features_previous["mean_down_payment"] = prev_application.groupby(by = "SK_ID_CURR")["RATE_DOWN_PAYMENT"].mean()
    #Sum (Mostly flags)
    features_previous["n_canceled"] = prev_application.groupby(by = "SK_ID_CURR")["flag_canceled"].sum()
    features_previous["n_refused"] = prev_application.groupby(by = "SK_ID_CURR")["flag_refused"].sum()
    features_previous["n_unused"] = prev_application.groupby(by = "SK_ID_CURR")["flag_unused"].sum()
    features_previous["n_down_payment"] = prev_application.groupby(by = "SK_ID_CURR")["flag_down_payment"].sum()
    features_previous["n_interest"] = prev_application.groupby(by = "SK_ID_CURR")["flag_interest"].sum()

    features_previous["n_last_apply_day"] = prev_application.groupby(by = "SK_ID_CURR")["NFLAG_LAST_APPL_IN_DAY"].sum()
    features_previous["n_insured_approval"] = prev_application.groupby(by = "SK_ID_CURR")["NFLAG_INSURED_ON_APPROVAL"].sum()
    features_previous["n_last_apply_contract"] = prev_application.groupby(by = "SK_ID_CURR")["FLAG_LAST_APPL_PER_CONTRACT"].sum()

#Categorical columns: most frequent feature used
    columns_cat = [
    'NAME_CONTRACT_TYPE',
    'WEEKDAY_APPR_PROCESS_START',
    'HOUR_APPR_PROCESS_START',
    'NAME_CASH_LOAN_PURPOSE',
    'NAME_CONTRACT_STATUS',
    'NAME_PAYMENT_TYPE',
    'CODE_REJECT_REASON',
    'NAME_TYPE_SUITE',
    'NAME_CLIENT_TYPE',
    'NAME_GOODS_CATEGORY',
    'NAME_PORTFOLIO',
    'NAME_PRODUCT_TYPE',
    'CHANNEL_TYPE',
    'SELLERPLACE_AREA',
    'NAME_SELLER_INDUSTRY',
    'NAME_YIELD_GROUP',
    'PRODUCT_COMBINATION',
    ]
    columns_freq = {
    'NAME_CONTRACT_TYPE' : "freq_contract_type",
    'WEEKDAY_APPR_PROCESS_START' : "freq_weekday",
    'HOUR_APPR_PROCESS_START' : "freq_hour",
    'NAME_CASH_LOAN_PURPOSE' : "freq_purpose",
    'NAME_CONTRACT_STATUS' : "freq_status",
    'NAME_PAYMENT_TYPE' : "freq_payment_type",
    'CODE_REJECT_REASON' : "freq_reject",
    'NAME_TYPE_SUITE' : "fred_suite",
    'NAME_CLIENT_TYPE' : "freq_client_type",
    'NAME_GOODS_CATEGORY' : "freq_goods",
    'NAME_PORTFOLIO' : "freq_portfolio",
    'NAME_PRODUCT_TYPE' : "freq_product",
    'CHANNEL_TYPE' : "freq_channel",
    'SELLERPLACE_AREA' : "freq_area",
    'NAME_SELLER_INDUSTRY' : "freq_industry",
    'NAME_YIELD_GROUP' : "freq_yield_group",
    'PRODUCT_COMBINATION' : "fred_product_comb",
}

    for col in columns_cat:
        features_previous[columns_freq[col]] =mode_column(df = prev_application,
                                                           col = col,
                                                           id_col = 'SK_ID_CURR',
                                                           cols_count = ['SK_ID_PREV'],
                                                            cols_sum = ['AMT_APPLICATION']
                                                         )
    

    #Payment installments
    
    prev_app_detail_instlmnt = (prev_application[["SK_ID_CURR", "SK_ID_PREV"]
                    ].drop_duplicates().set_index('SK_ID_CURR')
                     ).loc[features_previous.index].reset_index()
    prev_app_detail_instlmnt = prev_app_detail_instlmnt.merge(summarize_installments(prev_app_detail[["SK_ID_PREV"]]),
                     left_on = "SK_ID_PREV",
                     right_index=True)

    features_previous_instlmnt["n_installments"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR").n_versions.count()
    features_previous_instlmnt["max_installments"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR").n_versions.max()
    features_previous_instlmnt["min_installments"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR").n_versions.min()
    features_previous_instlmnt["latest_day"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["latest_day"].max()
    features_previous_instlmnt["first_day"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["first_day"].min()
    features_previous_instlmnt["latest_payment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["latest_payment"].max()
    features_previous_instlmnt["first_payment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["first_payment"].min()
    features_previous_instlmnt["max_installment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["max_installment"].max()
    features_previous_instlmnt["min_installment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["min_installment"].min()
    features_previous_instlmnt["mean_installment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["mean_installment"].mean()
    features_previous_instlmnt["mean_pay_pct_installment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["mean_pay_pct_installment"].mean()
    features_previous_instlmnt["min_pay_pct_installment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["min_pay_pct_installment"].min()
    features_previous_instlmnt["max_pay_pct_installment"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["max_pay_pct_installment"].max()
    features_previous_instlmnt["times_unfullfilled"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["times_unfullfilled"].sum()
    features_previous_instlmnt["times_unpayed"] = prev_app_detail_instlmnt.groupby(by = "SK_ID_CURR")["times_unpayed"].sum()

    #POS Cash
    

    prev_app_detail_cash = (prev_application[["SK_ID_CURR", "SK_ID_PREV"]
                    ].drop_duplicates().set_index('SK_ID_CURR')
                     ).loc[features_previous.index].reset_index()
    prev_app_detail_cash = prev_app_detail_cash.merge(summarize_cash_balance(prev_app_detail[["SK_ID_PREV"]],pos_cash_balance = pos_cash_balance),
                     left_on = "SK_ID_PREV",
                     right_index=True)
    features_previous_cash["total_months_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').n_months.sum()
    features_previous_cash["min_months_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').n_months.min()
    features_previous_cash["max_months_cash_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').n_months.max()
    features_previous_cash["total_prev_terms_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').total_prev_terms.sum()

    features_previous_cash["min_prev_term_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').min_prev_term.min()
    features_previous_cash["max_prev_term_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').max_prev_term.max()

    features_previous_cash["max_remaining_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').max_remaining.max()
    features_previous_cash["number_status_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').number_status.sum()

    features_previous_cash["total_dpd_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').total_dpd.sum()
    features_previous_cash["times_dpd_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').flag_dpd.sum()
    features_previous_cash["times_dpd_def_cash"] = prev_app_detail_cash.groupby(by = 'SK_ID_CURR').flag_dpd_def.sum()

    features_previous_cash["freq_latest_status_cash"] = mode_column(df = prev_app_detail_cash,
                                                           col = 'latest_status',
                                                           id_col = 'SK_ID_CURR',
                                                           cols_count = ['SK_ID_PREV'],
                                                            cols_sum = ['n_months']
                                                         )

    features_previous_cash["flag_cash"] = features_previous_cash["total_months_cash"].apply(lambda x: 1 if x>0 else 0)

    ##Credit cards
    prev_app_detail_cc = (prev_application[["SK_ID_CURR", "SK_ID_PREV"]
                    ].drop_duplicates().set_index('SK_ID_CURR')
                     ).loc[features_previous_cc.index].reset_index()
    prev_app_detail_cc = prev_app_detail_cc.merge(summarize_credit_card(prev_app_detail[["SK_ID_PREV"]]),
                     left_on = "SK_ID_PREV",
                     right_index=True)

    features_previous_cc["n_months_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["n_months_cc"].sum()

    features_previous_cc["mean_amt_balance_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_amt_balance"].mean()
    features_previous_cc["max_amt_balance_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_amt_balance"].max()
    features_previous_cc["min_amt_balance_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_amt_balance"].min()
    features_previous_cc["mean_amt_limit_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_amt_limit"].	mean()
    features_previous_cc["max_amt_limit_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_amt_limit"].max()	
    features_previous_cc["min_amt_limit_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_amt_limit"].min()	
    features_previous_cc["mean_amt_drawing_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_amt_drawing_current"].mean()	
    features_previous_cc["max_amt_drawing_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_amt_drawing_current"].max()	
    features_previous_cc["min_amt_drawing_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_amt_drawing_current"].min()	
    features_previous_cc["mean_amt_drawing_other_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_amt_drawing_other_current"].mean()	
    features_previous_cc["max_amt_drawing_other_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_amt_drawing_other_current"].max()	
    features_previous_cc["min_amt_drawing_other_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_amt_drawing_other_current"].min()	
    features_previous_cc["mean_amt_drawing_pos_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_amt_drawing_pos_current"].mean()	
    features_previous_cc["max_amt_drawing_pos_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_amt_drawing_pos_current"].max()	
    features_previous_cc["min_amt_drawing_pos_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_amt_drawing_pos_current"].min()	
    features_previous_cc["mean_min_installment_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_min_installment"].mean()	
    features_previous_cc["max_min_installment_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_min_installment"].max()	
    features_previous_cc["min_min_installment_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_min_installment"].min()	
    features_previous_cc["mean_payment_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_payment"].mean()	
    features_previous_cc["max_paymen_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_paymen"].max()	
    features_previous_cc["min_payment_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_payment"].min()	
    features_previous_cc["mean_payment_total_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_payment_total"].mean()	
    features_previous_cc["max_paymen_total_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_paymen_total"].max()	
    features_previous_cc["min_payment_total_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_payment_total"].min()	
    features_previous_cc["mean_receivable_principal_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_receivable_principal"].mean()	
    features_previous_cc["max_receivable_principal_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_receivable_principal"].max()	
    features_previous_cc["min_receivable_principal_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_receivable_principal"].min()	
    features_previous_cc["mean_receivable_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_receivable"].mean()	
    features_previous_cc["max_receivable_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_receivable"].max()	
    features_previous_cc["min_receivable_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_receivable"].min()	
    features_previous_cc["mean_receivable_total_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_receivable_total"].mean()	
    features_previous_cc["max_receivable_total_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_receivable_total"].max()	
    features_previous_cc["min_receivable_total_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_receivable_total"].min()	
    features_previous_cc["mean_drawings_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_drawings_current"].mean()	
    features_previous_cc["max_drawings_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_drawings_current"].max()	
    features_previous_cc["min_drawings_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_drawings_current"].min()	
    features_previous_cc["mean_drawings_other_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_drawings_other_current"].mean()	
    features_previous_cc["max_drawings_other_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_drawings_other_current"].max()	
    features_previous_cc["min_drawings_other_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_drawings_other_current"].min()	
    features_previous_cc["mean_drawings_pos_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_drawings_pos_current"].mean()	
    features_previous_cc["max_drawings_pos_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_drawings_pos_current"].max()	
    features_previous_cc["min_drawings_pos_current_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_drawings_pos_current"].min()	
    features_previous_cc["mean_installments_mature_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["mean_installments_mature"].mean()	
    features_previous_cc["max_installments_mature_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["max_installments_mature"].max()	
    features_previous_cc["min_installments_mature_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["min_installments_mature"].min()	
    features_previous_cc["total_dpd_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["total_dpd"].sum()	
    features_previous_cc["total_dpd_def_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["total_dpd_def"].sum()	
    features_previous_cc["flag_dpd_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["flag_dpd"].sum()
    features_previous_cc["flag_dpd_def_cc"] = prev_app_detail_cc.groupby(by = "SK_ID_CURR")["flag_dpd_def"].sum()	
    features_previous_cc["latest_status_cc"] = mode_column(df = prev_app_detail_cc,
                                                           col = 'latest_status',
                                                           id_col = 'SK_ID_CURR',
                                                           cols_count = ['SK_ID_PREV'],
                                                            cols_sum = ['max_amt_balance']
                                                         )
    features_previous = pd.concat([features_previous,features_previous_instlmnt,features_previous_cash,features_previous_cc])
    return features_previous.reset_index()
#

In [13]:
get_previous_application_features(prev_application[["SK_ID_CURR"]].sample(5),
                                 prev_application = prev_application)

NameError: name 'get_previous_application_features' is not defined

In [143]:
features_previous.head()

,previous_applications,n_types_applications,n_weekdays,n_hour_start,n_purpose,n_status,n_payment_type,n_code_rejection,n_suite,n_client_type,n_goods_category,n_portfolio,n_product_type,n_channel_type,n_sellerplace_area,n_seller_industry,n_yield_groups,n_product_comb,amt_annuity_mean,amt_annuity_min,amt_annuity_max,amt_annuity_median,amt_application_mean,amt_application_min,amt_application_max,amt_application_median,amt_credit_mean,amt_credit_min,amt_credit_max,amt_credit_median,amt_down_payment_mean,amt_down_payment_min,amt_down_payment_max,amt_down_payment_median,amt_goods_mean,amt_goods_min,amt_goods_max,amt_goods_median,cnt_payment_mean,cnt_payment_min,cnt_payment_max,cnt_payment_median,down_payment_mean,down_payment_min,down_payment_max,down_payment_median,rate_interested_primary_mean,rate_interested_primary_min,rate_interested_primary_max,rate_interested_primary_median,rate_intereseted_privileged_mean,rate_intereseted_privileged_min,rate_intereseted_privileged_max,rate_intereseted_privileged_median,days_decision_min,days_decision_max,days_first_drawing_min,days_first_drawing_max,days_first_due_min,days_first_due_max,days_last_due_first_min,days_last_due_first_max,days_last_due_min,days_last_due_max,days_termination_min,days_termination_max,mean_down_payment,n_canceled,n_refused,n_down_payment,n_interest,n_last_apply_day,n_insured_approval,n_last_apply_contract,freq_type,freq_weekday,freq_hour,freq_purpose,freq_status,freq_payment_type,freq_reject,fred_suite,freq_goods,freq_portfolio,freq_product,freq_channel,freq_area,freq_industry,freq_yield_group,fred_product_comb
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
215802,8,3,3,4,1,3,1,2,2,2,1,3,1,2,2,1,3,6,28955.512500,6750.000,50026.005,24465.9150,434727.562500,0.0,1485000.0,298910.25,473760.562500,0.0,1626133.5,324481.50,0.0,0.0,0.0,0.0,579636.750000,135000.0,1485000.0,337500.0,27.000000,0.0,48.0,33.0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1004,-330,365243.0,365243.0,-972.0,-300.0,-642.0,1110.0,-702.0,365243.0,-695.0,365243.0,0.000000,2,0,1,0,8,4.0,8,Repeater,WEDNESDAY,10,XAP,Approved,Cash through the bank,XAP,Family,Homewares,Cash,x-sell,Credit and cash offices,-1,Construction,middle,Cash X-Sell: middle
187440,6,1,4,5,1,2,1,2,2,3,2,1,0,2,5,1,2,2,7748.790000,4722.660,11406.780,7751.8800,51391.500000,41580.0,58896.0,54067.50,47765.250000,41580.0,56835.0,47144.25,3975.0,0.0,9000.0,2925.0,51391.500000,41580.0,58896.0,54067.5,8.333333,6.0,12.0,8.0,0.077018,0.000000,0.183580,0.056050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2749,-95,365243.0,365243.0,-2714.0,-143.0,-2564.0,127.0,-2624.0,-143.0,-2620.0,-135.0,0.077018,0,0,6,0,6,1.0,6,Repeater,TUESDAY,13,XAP,Refused,Cash through the bank,LIMIT,Unaccompanied,Mobile,POS,NaN,Country-wide,64,Connectivity,high,POS mobile with interest
135167,18,2,5,8,2,3,1,3,2,1,2,2,2,4,6,2,3,6,16867.882500,3458.520,43067.475,12000.7800,139665.750000,0.0,630000.0,51792.75,152691.000000,0.0,729567.0,47479.50,7731.0,3150.0,12006.0,7362.0,209498.625000,30708.0,630000.0,120024.0,15.166667,6.0,36.0,11.0,0.124883,0.101385,0.203378,0.110330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2799,-132,365243.0,365243.0,-2768.0,-451.0,-2618.0,239.0,-2618.0,365243.0,-2593.0,365243.0,0.124883,6,0,6,0,18,6.0,18,Repeater,FRIDAY,13,XAP,Approved,Cash through the bank,XAP,Unaccompanied,Mobile,Cash,x-sell,Credit and cash offices,-1,Connectivity,middle,Cash
290077,11,2,5,5,2,2,2,2,2,2,2,2,2,2,3,2,3,6,17961.876818,2332.440,52168.770,16099.0650,239080.909091,17550.0,1125000.0,180000.00,266460.545455,17100.0,1312110.0,203760.00,4038.0,1755.0,5859.0,4500.0,239080.909091,17550.0,1125000.0,180000.0,16.181818,6.0,36.0,12.0,0.113535,0.100000,0.139234,0.101371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2339,-225,365243.0,365243.0,-2259.0,-465.0,-1989.0,225.0,-2019.0,-315.0,-2011.0,-310.0,0.113535,0,0,3,0,11,4.0,11,Repeater,MONDAY,8,XAP,Approved,Non-cash from your account,XAP,Unaccompanied,Mobile,Cash,x-sell,Country-wide,39,Connec

# Bureau + Bureau Balance

In [123]:
bureau = pd.read_csv("data/bureau.csv")
bureau_balance = pd.read_csv("data/bureau_balance.csv")

In [207]:
def bureau_balance_summary(id_loans,
                          bureau_balance = bureau_balance):
    """We are working only with knowns"""
    # Flag if cancelled
    bureau_id = id_loans[["SK_ID_BUREAU"]].drop_duplicates().set_index("SK_ID_BUREAU")
    cancelled = bureau_balance[bureau_balance.STATUS=="C"][["SK_ID_BUREAU"]]
    cancelled["FLAG_CANCELLED"] = 1
    cancelled = cancelled.drop_duplicates()
    cancelled.set_index("SK_ID_BUREAU")
    bureau_id["flag_cancelled"] = cancelled.set_index("SK_ID_BUREAU")

    bureau_id["total_months_balance"] = bureau_balance.groupby(by = "SK_ID_BUREAU")["MONTHS_BALANCE"].count()

    # Number of times with DPD (Status > 1)
    bureau_balance.loc[bureau_balance["STATUS"].apply(lambda x:x in ['1', '2', '3', '5', '4']),"DPD_FLAG"] = "DPD" 
    bureau_id["months_dpd"] = bureau_balance[bureau_balance["STATUS"].apply(lambda x:x in ['1', '2', '3', '5', '4']
                                                                       )].groupby(by = "SK_ID_BUREAU")["MONTHS_BALANCE"].count()
# Number of times with DPD 31+ (Status > 2)
    bureau_id["months_dpd_31"] = bureau_balance[bureau_balance["STATUS"].apply(lambda x:x in [ '2', '3', '5', '4']
                                                                          )].groupby(by = "SK_ID_BUREAU")["MONTHS_BALANCE"].count()

    # Number of times with DPD 120+ (Status > 5)
    bureau_id["months_dpd_120"] = bureau_balance[bureau_balance["STATUS"].apply(lambda x:x in [ '5', ]
                                                                           )].groupby(by = "SK_ID_BUREAU")["MONTHS_BALANCE"].count()
    return bureau_id
    

In [253]:
def get_features_bureau(id_loans,
                                     bureau = bureau,
                                     bureau_balance = bureau_balance):
    features_bureau = id_loans[["SK_ID_CURR"]].set_index("SK_ID_CURR")
    features_bureau["n_total_credits"] = bureau.groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_active_credits"] = bureau[bureau["CREDIT_ACTIVE"]=='Active'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_closed_credits"] = bureau[bureau["CREDIT_ACTIVE"]=='Closed'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_sold_credits"] = bureau[bureau["CREDIT_ACTIVE"]=='Sold'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_bad_credits"] = bureau[bureau["CREDIT_ACTIVE"]=='Bad debt'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["freq_active"] = mode_column(
                                    df = bureau,
                                    col = 'CREDIT_ACTIVE',
                                    id_col = 'SK_ID_CURR',
                                    cols_count = ['SK_ID_BUREAU'],
                                    cols_sum = ['AMT_CREDIT_SUM_LIMIT']
                                            )
    features_bureau["n_credits_currency_1"] = bureau[bureau["CREDIT_CURRENCY"]=='currency 1'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_credits_currency_2"] = bureau[bureau["CREDIT_CURRENCY"]=='currency 2'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_credits_currency_3"] = bureau[bureau["CREDIT_CURRENCY"]=='currency 3'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_credits_currency_4"] = bureau[bureau["CREDIT_CURRENCY"]=='currency 4'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["freq_currency"] = mode_column(
                                    df = bureau,
                                    col = 'CREDIT_CURRENCY',
                                    id_col = 'SK_ID_CURR',
                                    cols_count = ['SK_ID_BUREAU'],
                                    cols_sum = ['AMT_CREDIT_SUM_LIMIT']
                                            )

    features_bureau["n_credits_consumer"] = bureau[bureau["CREDIT_TYPE"]=='Consumer credit'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_credits_car_loan"] = bureau[bureau["CREDIT_TYPE"]=='Car loan'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_credits_credit_card"] = bureau[bureau["CREDIT_TYPE"]=='Credit card'].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["n_credits_other"] = bureau[bureau["CREDIT_TYPE"].apply(lambda x: x not in ['Consumer credit','Credit card','Car loan'])
                                        ].groupby(by = "SK_ID_CURR" )["SK_ID_BUREAU"].count()
    features_bureau["freq_credit_type"] = mode_column(
                                    df = bureau,
                                    col = 'CREDIT_TYPE',
                                    id_col = 'SK_ID_CURR',
                                    cols_count = ['SK_ID_BUREAU'],
                                    cols_sum = ['AMT_CREDIT_SUM_LIMIT']
                                            )

    features_bureau["min_days_credits"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_CREDIT"].min()
    features_bureau["max_days_credit"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_CREDIT"].max()

    features_bureau["min_days_credit_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["CREDIT_DAY_OVERDUE"].min()
    features_bureau["flag_credit_overdue"] = features_bureau["min_days_credit_overdue"].apply(lambda x: 1 if x>0 else 0)
    features_bureau["max_days_credit_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["CREDIT_DAY_OVERDUE"].max()

    features_bureau["min_days_duration"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_CREDIT_ENDDATE"].min()
    features_bureau["max_days_duration"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_CREDIT_ENDDATE"].max()

    features_bureau["min_days_ended"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_ENDDATE_FACT"].min()
    features_bureau["max_days_ended"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_ENDDATE_FACT"].max()

    features_bureau["min_amt_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_MAX_OVERDUE"].min()
    features_bureau["max_amt_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_MAX_OVERDUE"].max()

    features_bureau["min_days_update"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_CREDIT_UPDATE"].min()
    features_bureau["max_days_update"] = bureau.groupby(by = "SK_ID_CURR" )["DAYS_CREDIT_UPDATE"].max()

    features_bureau["total_prolong"] = bureau.groupby(by = "SK_ID_CURR" )["CNT_CREDIT_PROLONG"].sum()

    features_bureau["mean_sum_debt"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_DEBT"].mean()
    features_bureau["max_sum_debt"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_DEBT"].max()
    features_bureau["min_sum_debt"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_DEBT"].min()
    features_bureau["median_sum_debt"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_DEBT"].median()

    features_bureau["mean_sum_limit"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_LIMIT"].mean()
    features_bureau["max_sum_limit"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_LIMIT"].max()
    features_bureau["min_sum_limit"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_LIMIT"].min()
    features_bureau["median_sum_limit"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_LIMIT"].median()

    features_bureau["mean_sum_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_OVERDUE"].mean()
    features_bureau["max_sum_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_OVERDUE"].max()
    features_bureau["min_sum_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_OVERDUE"].min()
    features_bureau["median_sum_overdue"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_CREDIT_SUM_OVERDUE"].median()

    features_bureau["mean_annuity"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_ANNUITY"].mean()
    features_bureau["max_annuity"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_ANNUITY"].max()
    features_bureau["min_annuity"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_ANNUITY"].min()
    features_bureau["median_annuity"] = bureau.groupby(by = "SK_ID_CURR" )["AMT_ANNUITY"].median()


    bureau_details = (bureau[["SK_ID_CURR", "SK_ID_BUREAU"]
                    ].drop_duplicates().set_index('SK_ID_CURR')
                     ).loc[features_bureau.index].reset_index()
    bureau_details = bureau_details.merge(bureau_balance_summary(bureau_details[["SK_ID_BUREAU"]]),
                     left_on = "SK_ID_BUREAU",
                     right_index=True)

    features_bureau["n_cancelled"] = bureau_details.groupby(by = "SK_ID_CURR")["flag_cancelled"].sum()

    features_bureau["tota_months_balance"] = bureau_details.groupby(by = "SK_ID_CURR")["total_months_balance"].sum()
    features_bureau["min_months_balance"] = bureau_details.groupby(by = "SK_ID_CURR")["total_months_balance"].min()
    features_bureau["max_months_balance"] = bureau_details.groupby(by = "SK_ID_CURR")["total_months_balance"].max()

    features_bureau["total_months_dpd"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd"].sum()
    features_bureau["min_months_dpd"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd"].min()
    features_bureau["max_months_dpd"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd"].max()

    features_bureau["total_months_dpd_31"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd_31"].sum()
    features_bureau["min_months_dpd_31"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd_31"].min()
    features_bureau["max_months_dpd_31"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd_31"].max()

    features_bureau["total_months_dpd_120"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd_120"].sum()
    features_bureau["min_months_dpd_120"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd_120"].min()
    features_bureau["max_months_dpd_120"] = bureau_details.groupby(by = "SK_ID_CURR")["months_dpd_120"].max()

    return features_bureau
			


With the features created and the features already from the application we create the following schema

In [7]:
data_ref = pd.read_csv("schema/features_schema_reference.csv")


In [396]:
def print_range(x):
    if pd.isna(x):
        return 'null'
    else:
        return str(x)
data_ref["range"] = data_ref.apply(lambda x: f"[{print_range(x['min_value'])}, {print_range(x['max_value'])}]", axis = 1)

In [11]:

data_schema = data_ref[["name", "type", "impute_value","range" ,"data_source"]].set_index("name").to_dict(orient = 'index')

In [12]:
import json

with open(SCHEMA_DIR /"data_schema.json", "w") as file:
    json.dump(data_schema, file, indent=4)  # indent=4 optional for pretty-printing


In [ ]:

from src.feature_extraction import get_previous_application_features, get_features_bureau
test_previous_data = get_previous_application_features(id_loans = test_raw[["SK_ID_CURR"]],
                                      prev_application = pd.read_csv(DATA_DIR / "previous_application.csv"),
                                      installments_payments = pd.read_csv(DATA_DIR / "installments_payments.csv"),
                                      pos_cash_balance = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv"),
                                      credit_card  = pd.read_csv(DATA_DIR / "credit_card_balance.csv"))
test_bureau =get_features_bureau(id_loans = test_raw[["SK_ID_CURR"]],
                   bureau = pd.read_csv(DATA_DIR / "bureau.csv"),
                   bureau_balance = pd.read_csv(DATA_DIR / "bureau_balance.csv"))
test_raw = test_raw.set_index("SK_ID_CURR")
test_final = pd.concat([test_raw,test_bureau,test_previous_data], axis = 1)
test_final.to_csv(DATA_DIR / "application_test_all_features.csv",
                 index = False)
train_previous_data = get_previous_application_features(id_loans = train_raw[["SK_ID_CURR"]],
                                      prev_application = pd.read_csv(DATA_DIR / "previous_application.csv"),
                                      installments_payments = pd.read_csv(DATA_DIR / "installments_payments.csv"),
                                      pos_cash_balance = pd.read_csv(DATA_DIR / "POS_CASH_balance.csv"),
                                      credit_card  = pd.read_csv(DATA_DIR / "credit_card_balance.csv"))
train_bureau =get_features_bureau(id_loans = train_raw[["SK_ID_CURR"]],
                   bureau = pd.read_csv(DATA_DIR / "bureau.csv"),
                   bureau_balance = pd.read_csv(DATA_DIR / "bureau_balance.csv"))
train_raw = train_raw.set_index("SK_ID_CURR")
train_final = pd.concat([test_raw,test_bureau,test_previous_data], axis = 1)
train_final.to_csv(DATA_DIR / "application_train_all_features.csv",
                  index = False)

In [ ]:
#schema {name : [kind,null, range, origin]}